# 0. Read files and Feature Engineering

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings

warnings.filterwarnings('ignore')

# Device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


In [2]:
# Load data
print("Loading data...")
train_df = pd.read_csv('trademaster25/train_v2.csv')
test_df = pd.read_csv('trademaster25/test_v2.csv')

print(f"Training set: {train_df.shape}")
print(f"Test set: {test_df.shape}")

# Feature and target columns
feature_cols = [f'feature_{i}' for i in range(1, 27)]+[f'feature_{j}' for j in range(28, 31)]
target_cols = ['target_short', 'target_medium', 'target_long']
time_cols = ['date_id', 'minute_id']
TARGET_WEIGHTS = {'short': 0.5, 'medium': 0.3, 'long': 0.2}

print(f"Features: {len(feature_cols)}")
print(f"Targets: {target_cols}")
print(f"Weights: {TARGET_WEIGHTS}")

Loading data...
Training set: (139392, 37)
Test set: (34348, 34)
Features: 29
Targets: ['target_short', 'target_medium', 'target_long']
Weights: {'short': 0.5, 'medium': 0.3, 'long': 0.2}


In [3]:
# Fill NaN with median and clip extreme values
for col in feature_cols:
    median_val = train_df[col].median()
    train_df[col].fillna(median_val, inplace=True)
    test_df[col].fillna(median_val, inplace=True)

    # Clip extreme values at 1% and 99% quantiles
    lower = train_df[col].quantile(0.01)
    upper = train_df[col].quantile(0.99)
    train_df[col] = train_df[col].clip(lower, upper)
    test_df[col] = test_df[col].clip(lower, upper)

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# # 可视化第一个特征的分布变化
# plt.figure(figsize=(10, 4))
# sns.histplot(train_df['feature_1'], kde=True)
# plt.title('Feature 1 (Before GaussRank)')
# plt.show()

from sklearn.preprocessing import QuantileTransformer

print("4.1.2 GaussRank 归一化 (Rank-Gauss)")
print("=" * 80)

# 使用 QuantileTransformer 进行 GaussRank 归一化
# output_distribution='normal' 将数据映射为高斯分布
gauss_scaler = QuantileTransformer(output_distribution='normal', random_state=42)

print("Applying GaussRank normalization...")
# 对特征列进行转换
train_df[feature_cols] = gauss_scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = gauss_scaler.transform(test_df[feature_cols])

print("GaussRank normalization complete.")

# 验证归一化后的分布
print("\n归一化后的偏度和峰度 (前5个特征):")
print(train_df[feature_cols[:5]].agg(['skew', 'kurtosis']))

# # 可视化第一个特征的分布变化
# plt.figure(figsize=(10, 4))
# sns.histplot(train_df['feature_1'], kde=True)
# plt.title('Feature 1 (After GaussRank)')
# plt.show()

4.1.2 GaussRank 归一化 (Rank-Gauss)
Applying GaussRank normalization...
GaussRank normalization complete.

归一化后的偏度和峰度 (前5个特征):
          feature_1  feature_2  feature_3  feature_4  feature_5
skew       2.654684   0.010780  -0.031259   0.000651  -0.008955
kurtosis   5.047422   5.505653   5.551395   5.514407   5.330535


# 1. KNN+SAE+CNN+Embeddeing XGB

## 1.1 purged Group time split

In [ ]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
import pickle

print("4.1.3 Purged Group Time Series Split (核心验证策略)")
print("=" * 80)

class PurgedGroupTimeSeriesSplit:
    def __init__(self, n_splits=5, gap=0):
        self.n_splits = n_splits
        self.gap = gap

    def split(self, X, y=None, groups=None):
        if groups is None:
            tscv = TimeSeriesSplit(n_splits=self.n_splits)
            for train_idx, test_idx in tscv.split(X):
                test_start = test_idx[0]
                train_end_limit = test_start - self.gap
                train_idx_purged = train_idx[train_idx <= train_end_limit]
                if len(train_idx_purged) > 0:
                    yield train_idx_purged, test_idx
        else:
            unique_groups = np.unique(groups)
            unique_groups.sort()
            tscv = TimeSeriesSplit(n_splits=self.n_splits)
            for train_groups_idx, test_groups_idx in tscv.split(unique_groups):
                train_groups = unique_groups[train_groups_idx]
                test_groups = unique_groups[test_groups_idx]
                
                test_start_group_idx = test_groups_idx[0]
                train_end_group_idx_limit = test_start_group_idx - self.gap
                train_groups_idx_purged = train_groups_idx[train_groups_idx <= train_end_group_idx_limit]
                
                if len(train_groups_idx_purged) > 0:
                    train_groups_purged = unique_groups[train_groups_idx_purged]
                    train_idx = np.where(np.isin(groups, train_groups_purged))[0]
                    test_idx = np.where(np.isin(groups, test_groups))[0]
                    yield train_idx, test_idx

# --- 策略配置 ---
# 1. Grouping: 使用 date_id (天)
#    理由: 即使有 minutes_id，按天分组能更彻底地防止日内微观结构噪音的泄露，且计算效率更高。
# 2. Gap: 取决于预测目标的最长跨度
#    理由: 如果同时预测"几分钟后"(Short)和"几天后"(Long)，必须使用覆盖 Long 的 Gap。
#    例如: Long 是预测 5 天后的收益率，那么 Gap 至少要设为 5 (天)。
#    虽然这对 Short 目标来说 Gap 过大（浪费了一些近期数据），但能确保验证集的绝对安全（无泄露）。

N_SPLITS = 5
# 假设 date_id 是连续的整数代表天数
# 假设 target_long 预测未来 5 天，则设置 GAP = 5 + 1 (缓冲) = 6
GAP = 6 

# 检查并设置 Groups
groups = None
if 'date_id' in train_df.columns:
    groups = train_df['date_id'].values
    print(f"✅ 使用 'date_id' 进行分组 (Groups: {len(np.unique(groups))} days)")
    print(f"✅ 设置 Gap = {GAP} (days) 以覆盖最长预测窗口")
else:
    print("no 'date_id' column found for grouping. Using standard TimeSeriesSplit.")

cv = PurgedGroupTimeSeriesSplit(n_splits=N_SPLITS, gap=GAP)

# 生成并保存分割
splits = []
for train_idx, test_idx in cv.split(train_df, groups=groups):
    splits.append((train_idx, test_idx))

with open('cv_splits.pkl', 'wb') as f:
    pickle.dump(splits, f)

print(f"\n已生成 {len(splits)} 折交叉验证分割。")
print(f"分割方案已保存至: cv_splits.pkl")

# 打印第一折的详细信息用于检查
if len(splits) > 0:
    train_idx_0, test_idx_0 = splits[0]
    print("\n--- Fold 1 详情 ---")
    print(f"Train samples: {len(train_idx_0):,}")
    print(f"Test samples:  {len(test_idx_0):,}")
    if groups is not None:
        print(f"Train dates: {groups[train_idx_0].min()} -> {groups[train_idx_0].max()}")
        print(f"Test dates:  {groups[test_idx_0].min()} -> {groups[test_idx_0].max()}")
        print(f"Gap 验证: Test Start ({groups[test_idx_0].min()}) - Train End ({groups[train_idx_0].max()}) = {groups[test_idx_0].min() - groups[train_idx_0].max()}")

## 1.2 KNN Feature Engineering

In [ ]:
import cudf
import cuml
from cuml.neighbors import NearestNeighbors
from cuml.preprocessing import StandardScaler as cuStandardScaler
import numpy as np
import pandas as pd
import gc
import pickle

print("4.2 阶段二：RAPIDS 加速的特征工程 (Train + Test)")
print("=" * 80)

# 准备数据 (转换为 GPU DataFrame)
print("Preparing data for GPU...")
X_all = train_df[feature_cols].values
y_all = train_df[target_cols].values
X_test_all = test_df[feature_cols].values # Test set

# 定义 KNN 参数
K_VALUES = [5, 10, 20]
METRICS = ['euclidean', 'cosine']

# 初始化结果容器
n_samples = len(train_df)
n_test_samples = len(test_df)
n_knn_features = len(K_VALUES) * len(METRICS) * (1 + len(target_cols)) 
knn_feature_names = []
knn_features = np.zeros((n_samples, n_knn_features), dtype=np.float32)
knn_features_test = np.zeros((n_test_samples, n_knn_features), dtype=np.float32)

print(f"Generating {n_knn_features} KNN features...")

# 加载之前保存的 splits
with open('cv_splits.pkl', 'rb') as f:
    splits = pickle.load(f)

knn_features[:] = np.nan

# --- Part 1: Generate KNN features for Train (CV) ---
for fold, (train_idx, val_idx) in enumerate(splits):
    print(f"\n--- Processing Fold {fold + 1}/{len(splits)} (Train CV) ---")
    
    # 数据移至 GPU
    X_train_gpu = cudf.DataFrame(X_all[train_idx], columns=feature_cols)
    X_val_gpu = cudf.DataFrame(X_all[val_idx], columns=feature_cols)
    
    # 标准化 (Fit on Train, Transform on Val)
    scaler = cuStandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_gpu)
    X_val_scaled = scaler.transform(X_val_gpu)
    
    col_idx = 0
    
    for k in K_VALUES:
        for metric in METRICS:
            # Build KNN model
            model = NearestNeighbors(n_neighbors=k, metric=metric)
            model.fit(X_train_scaled)
            
            # Get distances and indices
            distances, indices = model.kneighbors(X_val_scaled)
            
            # 1. Feature: Mean Distance
            dist_mean = distances.mean(axis=1)
            
            feat_name_dist = f'knn_k{k}_{metric}_dist_mean'
            if fold == 0: knn_feature_names.append(feat_name_dist)
            
            knn_features[val_idx, col_idx] = dist_mean.to_numpy()
            col_idx += 1
            
            # 2. Feature: Mean Target of Neighbors
            indices_cpu = indices.to_numpy()
            y_train_cpu = y_all[train_idx] 
            
            for t_i, target_col in enumerate(target_cols):
                neighbor_targets = y_train_cpu[indices_cpu, t_i]
                target_mean = neighbor_targets.mean(axis=1)
                
                feat_name_target = f'knn_k{k}_{metric}_{target_col}_mean'
                if fold == 0: knn_feature_names.append(feat_name_target)
                
                knn_features[val_idx, col_idx] = target_mean
                col_idx += 1
            
    del X_train_gpu, X_val_gpu, X_train_scaled, X_val_scaled
    gc.collect()

# --- Part 2: Generate KNN features for Test (Full Train) ---
print("\n--- Processing Test Set (Full Train Context) ---")
X_train_full_gpu = cudf.DataFrame(X_all, columns=feature_cols)
X_test_gpu = cudf.DataFrame(X_test_all, columns=feature_cols)

scaler_full = cuStandardScaler()
X_train_full_scaled = scaler_full.fit_transform(X_train_full_gpu)
X_test_scaled = scaler_full.transform(X_test_gpu)

col_idx = 0
for k in K_VALUES:
    for metric in METRICS:
        model = NearestNeighbors(n_neighbors=k, metric=metric)
        model.fit(X_train_full_scaled)
        
        distances, indices = model.kneighbors(X_test_scaled)
        
        # 1. Dist Mean
        dist_mean = distances.mean(axis=1)
        knn_features_test[:, col_idx] = dist_mean.to_numpy()
        col_idx += 1
        
        # 2. Target Mean
        indices_cpu = indices.to_numpy()
        y_train_cpu = y_all # Full labels
        
        for t_i, target_col in enumerate(target_cols):
            neighbor_targets = y_train_cpu[indices_cpu, t_i]
            target_mean = neighbor_targets.mean(axis=1)
            knn_features_test[:, col_idx] = target_mean
            col_idx += 1

del X_train_full_gpu, X_test_gpu, X_train_full_scaled, X_test_scaled
gc.collect()

# 保存 KNN 特征
knn_df = pd.DataFrame(knn_features, columns=knn_feature_names)
knn_df = knn_df.fillna(knn_df.mean())

knn_test_df = pd.DataFrame(knn_features_test, columns=knn_feature_names)
knn_test_df = knn_test_df.fillna(knn_test_df.mean()) # Should not have NaNs usually

print(f"\nKNN Features Shape (Train): {knn_df.shape}")
print(f"KNN Features Shape (Test): {knn_test_df.shape}")

knn_df.to_parquet('knn_features.parquet')
knn_test_df.to_parquet('knn_features_test.parquet')
print("✅ KNN features saved to 'knn_features.parquet' and 'knn_features_test.parquet'")

# 合并回主 DataFrame (Train only for DAE training, but we need Test for DAE inference)
train_df_with_knn = pd.concat([train_df, knn_df], axis=1)
test_df_with_knn = pd.concat([test_df, knn_test_df], axis=1)
all_feature_cols = feature_cols + knn_feature_names
print(f"Total features for DAE: {len(all_feature_cols)}")

## 1.3 DAE Feature Engineering

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

print("4.3 阶段三：DAE 潜在特征提取 (Train + Test)")
print("=" * 80)

# DAE Dataset with Swap Noise
class DAEDataset(Dataset):
    def __init__(self, X, noise_prob=0.15, training=True):
        self.X = torch.FloatTensor(X)
        self.noise_prob = noise_prob
        self.n_features = X.shape[1]
        self.training = training
        
    def train(self):
        self.training = True
        
    def eval(self):
        self.training = False
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        
        # Swap Noise
        if self.noise_prob > 0 and self.training:
            mask = torch.rand(self.n_features) < self.noise_prob
            if mask.any():
                noise_idx = torch.randint(0, len(self.X), (1,)).item()
                x[mask] = self.X[noise_idx][mask]
                
        return x, self.X[idx] 

# Bottleneck DAE Model
class DAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, latent_dim=64):
        super(DAE, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Linear(128, latent_dim),
            nn.BatchNorm1d(latent_dim), 
            nn.SiLU() 
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Linear(128, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, input_dim)
        )
        
    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent

# 准备数据
X_dae = train_df_with_knn[all_feature_cols].fillna(0).values
X_dae_test = test_df_with_knn[all_feature_cols].fillna(0).values # Test Data

print(f"DAE Input Shape (Train): {X_dae.shape}")
print(f"DAE Input Shape (Test): {X_dae_test.shape}")

# Dataloader
BATCH_SIZE = 512
train_dataset_dae = DAEDataset(X_dae, noise_prob=0.15, training=True)
train_loader_dae = DataLoader(train_dataset_dae, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# 初始化模型
INPUT_DIM = X_dae.shape[1]
dae_model = DAE(input_dim=INPUT_DIM).to(DEVICE)

# 训练配置
optimizer = optim.AdamW(dae_model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()
EPOCHS = 70
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, steps_per_epoch=len(train_loader_dae), epochs=EPOCHS
)

print("\nStarting DAE Training...")
dae_model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for x_noisy, x_clean in train_loader_dae:
        x_noisy, x_clean = x_noisy.to(DEVICE), x_clean.to(DEVICE)
        
        optimizer.zero_grad()
        reconstructed, _ = dae_model(x_noisy)
        loss = criterion(reconstructed, x_clean)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader_dae)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.6f}")

print("DAE Training Complete.")

# 提取潜在特征 (Latent Features) - Train
print("\nExtracting Latent Features (Train)...")
dae_model.eval()
latent_features = []
extract_dataset = DAEDataset(X_dae, noise_prob=0.0, training=False)
extract_loader = DataLoader(extract_dataset, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for x, _ in extract_loader:
        x = x.to(DEVICE)
        _, latent = dae_model(x)
        latent_features.append(latent.cpu().numpy())

latent_features = np.vstack(latent_features)
dae_feat_cols = [f'dae_{i}' for i in range(latent_features.shape[1])]
dae_df = pd.DataFrame(latent_features, columns=dae_feat_cols)

# 提取潜在特征 (Latent Features) - Test
print("Extracting Latent Features (Test)...")
latent_features_test = []
extract_dataset_test = DAEDataset(X_dae_test, noise_prob=0.0, training=False)
extract_loader_test = DataLoader(extract_dataset_test, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for x, _ in extract_loader_test:
        x = x.to(DEVICE)
        _, latent = dae_model(x)
        latent_features_test.append(latent.cpu().numpy())

latent_features_test = np.vstack(latent_features_test)
dae_test_df = pd.DataFrame(latent_features_test, columns=dae_feat_cols)

print(f"Latent Features Shape (Train): {dae_df.shape}")
print(f"Latent Features Shape (Test): {dae_test_df.shape}")

dae_df.to_parquet('dae_features.parquet')
dae_test_df.to_parquet('dae_features_test.parquet')
print("✅ DAE features saved to 'dae_features.parquet' and 'dae_features_test.parquet'")

## 1.4 SAE Feature Engineering

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

print("4.4 阶段四：SAE 有监督自编码器 (Supervised Autoencoder)")
print("=" * 80)

# 1. 定义 SAE 数据集
class SAEDataset(Dataset):
    def __init__(self, X, y, training=True):
        self.X = torch.FloatTensor(X)
        # 如果 y 是连续值，转换为二分类目标 (y > 0)
        # 这里假设 y_all 是收益率，我们需要预测涨跌
        # 注意：如果 y 已经是 0/1 标签，则不需要转换
        if len(np.unique(y)) > 2:
             self.y = torch.FloatTensor((y > 0).astype(int))
        else:
             self.y = torch.FloatTensor(y)
             
        self.training = training
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 2. 定义 SAE 模型
class SAE(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=128, latent_dim=32, dropout_rate=0.1):
        super(SAE, self).__init__()
        
        # 编码器 (Encoder)
        # Dense -> BN -> Swish (SiLU)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.SiLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dim // 2, latent_dim),
            nn.BatchNorm1d(latent_dim),
            nn.SiLU()
        )
        
        # 解码器 (Reconstruction Stream)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.SiLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dim, input_dim)
        )
        
        # 预测头 (Prediction Stream)
        # 残差连接 (Skip Connection): 原始输入 + 瓶颈层输出 -> 预测
        self.head = nn.Sequential(
            nn.Linear(latent_dim + input_dim, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, output_dim),
            nn.Sigmoid() # BCE Loss 需要输出在 [0, 1]
        )
        
    def forward(self, x, training=False):
        # 噪声注入 (Gaussian Noise Injection)
        if training:
            noise = torch.randn_like(x) * 0.1 # 噪声强度可调
            x_encoded = x + noise
        else:
            x_encoded = x
            
        latent = self.encoder(x_encoded)
        reconstructed = self.decoder(latent)
        
        # Skip Connection: 拼接 Latent 和 原始 Input
        combined = torch.cat([latent, x], dim=1)
        prediction = self.head(combined)
        
        return reconstructed, prediction, latent

# 3. 准备数据
# 使用之前准备好的 X_dae (包含原始特征 + KNN特征)
# y_all 是目标变量
X_sae = X_dae
y_sae = y_all 
X_sae_test = X_dae_test

print(f"SAE Input Shape: {X_sae.shape}")
print(f"SAE Target Shape: {y_sae.shape}")

# DataLoader
BATCH_SIZE = 512
train_dataset_sae = SAEDataset(X_sae, y_sae, training=True)
train_loader_sae = DataLoader(train_dataset_sae, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# 4. 初始化模型
INPUT_DIM = X_sae.shape[1]
OUTPUT_DIM = y_sae.shape[1]
LATENT_DIM = 32 # 建议 16 或 32

sae_model = SAE(input_dim=INPUT_DIM, output_dim=OUTPUT_DIM, latent_dim=LATENT_DIM).to(DEVICE)

# 5. 训练配置
optimizer = optim.AdamW(sae_model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion_recon = nn.MSELoss()
criterion_pred = nn.BCELoss()

EPOCHS = 50
LAMBDA = 0.5 # 损失权重: (1 - lambda) * Recon + lambda * Pred

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, steps_per_epoch=len(train_loader_sae), epochs=EPOCHS
)

print("\nStarting SAE Training...")
sae_model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    total_recon_loss = 0
    total_pred_loss = 0
    
    for x, y in train_loader_sae:
        x, y = x.to(DEVICE), y.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Forward pass
        reconstructed, prediction, _ = sae_model(x, training=True)
        
        # Calculate losses
        loss_recon = criterion_recon(reconstructed, x)
        loss_pred = criterion_pred(prediction, y)
        
        # Combined loss
        loss = (1 - LAMBDA) * loss_recon + LAMBDA * loss_pred
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        total_recon_loss += loss_recon.item()
        total_pred_loss += loss_pred.item()
        
    avg_loss = total_loss / len(train_loader_sae)
    avg_recon = total_recon_loss / len(train_loader_sae)
    avg_pred = total_pred_loss / len(train_loader_sae)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.6f} | Recon: {avg_recon:.6f} | Pred: {avg_pred:.6f}")

print("SAE Training Complete.")

# 6. 提取潜在特征 (Latent Features)
print("\nExtracting SAE Latent Features...")
sae_model.eval()

# Train
latent_features_sae = []
extract_dataset_sae = SAEDataset(X_sae, y_sae, training=False)
extract_loader_sae = DataLoader(extract_dataset_sae, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for x, _ in extract_loader_sae:
        x = x.to(DEVICE)
        _, _, latent = sae_model(x, training=False)
        latent_features_sae.append(latent.cpu().numpy())

latent_features_sae = np.vstack(latent_features_sae)
sae_feat_cols = [f'sae_{i}' for i in range(latent_features_sae.shape[1])]
sae_df = pd.DataFrame(latent_features_sae, columns=sae_feat_cols)

# Test
latent_features_sae_test = []
# For test set, we don't have targets usually, or we don't use them for extraction
# We can reuse SAEDataset but pass dummy targets or modify dataset
# Here we just pass dummy targets for simplicity
dummy_y_test = np.zeros((len(X_sae_test), OUTPUT_DIM))
extract_dataset_sae_test = SAEDataset(X_sae_test, dummy_y_test, training=False)
extract_loader_sae_test = DataLoader(extract_dataset_sae_test, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for x, _ in extract_loader_sae_test:
        x = x.to(DEVICE)
        _, _, latent = sae_model(x, training=False)
        latent_features_sae_test.append(latent.cpu().numpy())

latent_features_sae_test = np.vstack(latent_features_sae_test)
sae_test_df = pd.DataFrame(latent_features_sae_test, columns=sae_feat_cols)

print(f"SAE Latent Features Shape (Train): {sae_df.shape}")
print(f"SAE Latent Features Shape (Test): {sae_test_df.shape}")

sae_df.to_parquet('sae_features.parquet')
sae_test_df.to_parquet('sae_features_test.parquet')
print("✅ SAE features saved to 'sae_features.parquet' and 'sae_features_test.parquet'")

## 1.5 CNN Feature Engineering

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import gc

print("4.5 阶段五：1D-CNN 时序特征提取 (Causal Dilated Convolution)")
print("=" * 80)

# --- 1. 数据重塑 (Reshaping) ---
# 目标: 将表格数据转换为 (Batch, Sequence_Length=240, Features)
# 假设数据已经按 (date_id, time_id/feature_6, stock_id) 排序，或者我们需要手动排序
# 这里我们假设每一天 (date_id) 对每个股票 (stock_id) 都有完整的 240 个时间步 (feature_6)
# 如果数据不完整，我们需要填充 (Padding)

def reshape_to_sequence(df, feature_cols, seq_len=240, fill_value=0):
    """
    将 DataFrame 重塑为 (N_samples, Seq_Len, N_features)
    注意：这通常需要数据按 (Date, Stock, Time) 组织。
    如果数据是混合的，我们需要先 Pivot。
    """
    print("Reshaping data to sequences...")
    
    # 检查必要的列
    if 'date_id' not in df.columns or 'feature_6' not in df.columns:
        print("⚠️ Warning: 'date_id' or 'feature_6' (time) missing. Cannot strictly reshape by time.")
        print("Assuming data is already sorted and can be chunked (Risk of misalignment).")
        # 简单粗暴的 Reshape (仅用于演示，实际需谨慎)
        # 丢弃多余的行以匹配 seq_len
        n_rows = len(df)
        n_samples = n_rows // seq_len
        n_keep = n_samples * seq_len
        
        data = df[feature_cols].values[:n_keep]
        reshaped = data.reshape(n_samples, seq_len, len(feature_cols))
        return reshaped, df.index[:n_keep:seq_len] # 返回对应的索引(每段的开始)

    # 严谨的做法: Pivot table
    # 假设 feature_6 是 0-239 的整数
    # 我们需要构建一个 MultiIndex (date_id, stock_id) -> columns (feature_6)
    # 但由于特征有多个，这会变成 (N_groups, Seq_Len, N_features)
    
    # 1. 确定分组键
    group_keys = ['date_id']
    if 'stock_id' in df.columns:
        group_keys.append('stock_id')
    
    # 2. 排序
    print("Sorting data...")
    df_sorted = df.sort_values(by=group_keys + ['feature_6'])
    
    # 3. 提取 Values
    # 这种方法要求每个组严格有 240 行。如果不是，需要更复杂的处理 (Pad/Truncate)
    # 这里为了演示核心架构，我们采用一种高效的近似方法：
    # 直接按顺序切分，假设数据大部分是完整的。
    
    # 更好的方法：使用 GroupBy + Apply (慢) 或 Pivot (内存大)
    # 鉴于内存限制，我们尝试一种基于数组的方法
    
    values = df_sorted[feature_cols].values
    # 假设每个组都是连续的，我们通过 date_id 和 stock_id 的变化来切分
    # 但为了代码能跑通且不OOM，我们这里假设数据预处理已经保证了每天每股有记录
    # 或者我们只使用滑动窗口 (Rolling Window) ?
    # 题目要求 "利用240个时间步"，通常指日内完整序列。
    
    # 简化方案：
    # 假设数据集中大部分是完整的日内数据。
    # 我们将数据 Pad 到 240 的倍数 (或截断)
    n_features = len(feature_cols)
    n_total = len(df_sorted)
    n_samples = n_total // seq_len
    
    print(f"Total rows: {n_total}, Seq Len: {seq_len}, Resulting Samples: {n_samples}")
    
    # 截断多余部分
    values = values[:n_samples * seq_len]
    reshaped = values.reshape(n_samples, seq_len, n_features)
    
    return reshaped, df_sorted.index[:n_samples*seq_len:seq_len]

# 使用所有特征 (Raw + KNN + SAE Latent if available, but here we use Raw+KNN)
# 注意：通常 CNN 输入原始特征即可，不需要太高级的特征，因为它自己会提取
cnn_input_cols = feature_cols # + knn_feature_names (可选)
X_seq_train, _ = reshape_to_sequence(train_df, cnn_input_cols)
X_seq_test, _ = reshape_to_sequence(test_df, cnn_input_cols)

# 对应的 Target 也需要 Reshape
# 我们通常预测序列的最后一个点，或者整个序列
# 这里假设我们预测序列中每一步的收益，或者只取最后一步作为该序列的代表
# 为了简单，我们取每个序列的平均 Target 或最后一个 Target
# 但更高级的做法是 Sequence-to-Sequence 预测
y_seq_train, _ = reshape_to_sequence(train_df, target_cols)
# y_seq_train shape: (N_samples, 240, 3)

print(f"CNN Input Shape (Train): {X_seq_train.shape}")
print(f"CNN Input Shape (Test): {X_seq_test.shape}")

# --- 2. 模型定义 (Causal Dilated CNN) ---

class CausalConv1d(nn.Module):
    """
    因果卷积层：确保 t 时刻的输出只依赖于 t 及之前的输入。
    通过不对称 Padding 实现。
    """
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super(CausalConv1d, self).__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels, 
            out_channels, 
            kernel_size, 
            padding=self.padding, 
            dilation=dilation
        )
        
    def forward(self, x):
        # x: (Batch, Channels, Seq_Len)
        # Conv1d output with padding will be: (Batch, Out_Channels, Seq_Len + Padding)
        # We need to slice off the extra padding from the end to keep causality
        x = self.conv(x)
        if self.padding > 0:
            x = x[:, :, :-self.padding]
        return x

class DilatedCNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=3, seq_len=240):
        super(DilatedCNN, self).__init__()
        
        self.seq_len = seq_len
        
        # 初始投影
        self.start_conv = nn.Conv1d(input_dim, hidden_dim, kernel_size=1)
        
        # 膨胀卷积块 (WaveNet style)
        # Dilation rates: 1, 2, 4, 8, 16, 32
        self.layers = nn.ModuleList()
        dilations = [1, 2, 4, 8, 16, 32]
        
        for d in dilations:
            self.layers.append(
                nn.Sequential(
                    CausalConv1d(hidden_dim, hidden_dim, kernel_size=3, dilation=d),
                    nn.BatchNorm1d(hidden_dim),
                    nn.SiLU(), # Swish
                    nn.Dropout(0.1)
                )
            )
            
        # 残差连接处理 (1x1 Conv for skip connections if needed, here simple addition)
        
        # 最终预测头
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.SiLU(),
            nn.Linear(64, output_dim)
        )
        
    def forward(self, x):
        # x: (Batch, Seq_Len, Features) -> Permute to (Batch, Features, Seq_Len)
        x = x.permute(0, 2, 1)
        
        x = self.start_conv(x)
        
        skip_connections = []
        for layer in self.layers:
            residual = x
            x = layer(x)
            x = x + residual # Residual connection
            skip_connections.append(x)
            
        # 聚合所有层的输出 (Skip connections sum)
        # x = sum(skip_connections) 
        
        # Permute back to (Batch, Seq_Len, Hidden)
        x = x.permute(0, 2, 1)
        
        # Prediction
        out = self.head(x)
        
        # Return: Output (Batch, Seq_Len, Target_Dim), Features (Batch, Seq_Len, Hidden)
        return out, x

# --- 3. 训练循环 ---

class CNNDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y) if y is not None else None
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# Config
BATCH_SIZE = 128 # Smaller batch size for sequences
train_dataset_cnn = CNNDataset(X_seq_train, y_seq_train)
train_loader_cnn = DataLoader(train_dataset_cnn, batch_size=BATCH_SIZE, shuffle=True)

cnn_model = DilatedCNN(input_dim=len(cnn_input_cols)).to(DEVICE)
optimizer = optim.AdamW(cnn_model.parameters(), lr=1e-3)
criterion = nn.MSELoss() # Regression for targets

EPOCHS = 30
print("\nStarting 1D-CNN Training...")
cnn_model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for batch in train_loader_cnn:
        if len(batch) == 2:
            x, y = batch
            x, y = x.to(DEVICE), y.to(DEVICE)
        else:
            continue # Should not happen
            
        optimizer.zero_grad()
        preds, _ = cnn_model(x)
        
        # Loss calculation
        # We can calculate loss on all time steps or just the last one
        # Here: All time steps (Sequence-to-Sequence)
        loss = criterion(preds, y)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss / len(train_loader_cnn):.6f}")

print("1D-CNN Training Complete.")

# --- 4. 特征提取 ---
print("\nExtracting CNN Features...")
cnn_model.eval()

# Helper to extract and flatten
def extract_cnn_features(X_seq):
    dataset = CNNDataset(X_seq)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    features_list = []
    
    with torch.no_grad():
        for x in loader:
            x = x.to(DEVICE)
            _, feats = cnn_model(x) # (Batch, Seq_Len, Hidden)
            features_list.append(feats.cpu().numpy())
            
    # Stack and Flatten: (N_samples, Seq_Len, Hidden) -> (N_samples * Seq_Len, Hidden)
    features_array = np.vstack(features_list)
    n_samples, seq_len, hidden_dim = features_array.shape
    flattened = features_array.reshape(n_samples * seq_len, hidden_dim)
    return flattened

# Extract
cnn_feats_train = extract_cnn_features(X_seq_train)
cnn_feats_test = extract_cnn_features(X_seq_test)

# Create DataFrame
# Note: We need to be careful about alignment. 
# The reshaping logic truncated some rows. We need to align with the truncated index.
# For simplicity in this demo, we just save what we have.
cnn_feat_cols = [f'cnn_{i}' for i in range(cnn_feats_train.shape[1])]

cnn_df = pd.DataFrame(cnn_feats_train, columns=cnn_feat_cols)
cnn_test_df = pd.DataFrame(cnn_feats_test, columns=cnn_feat_cols)

print(f"CNN Features Shape (Train): {cnn_df.shape}")
print(f"CNN Features Shape (Test): {cnn_test_df.shape}")

cnn_df.to_parquet('cnn_features.parquet')
cnn_test_df.to_parquet('cnn_features_test.parquet')
print("✅ CNN features saved to 'cnn_features.parquet' and 'cnn_features_test.parquet'")


## 1.6 FTTransformer Opt

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print("4.4 阶段四：FT-Transformer 深度优化")
print("=" * 80)

class ReGLU(nn.Module):
    def forward(self, x):
        a, b = x.chunk(2, dim=-1)
        return a * F.relu(b)

class FeatureTokenizer(nn.Module):
    def __init__(self, n_num_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_num_features, d_token))
        self.bias = nn.Parameter(torch.randn(n_num_features, d_token))
        
    def forward(self, x):
        # x: (batch, n_features)
        # out: (batch, n_features, d_token)
        x = x.unsqueeze(-1) * self.weight + self.bias
        return x

class FTTransformer(nn.Module):
    def __init__(
        self, 
        n_num_features, 
        n_targets, 
        d_token=192, 
        n_layers=4, #3
        n_heads=8, 
        d_ffn_factor=1.33, 
        attention_dropout=0.2, 
        ffn_dropout=0.1,
        residual_dropout=0.0,
    ):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num_features, d_token)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token))
        
        self.layers = nn.ModuleList([])
        # ReGLU input size is d_ffn * 2, output is d_ffn
        d_ffn = int(d_token * d_ffn_factor * 2 / 3) * 2 
        
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                'norm1': nn.LayerNorm(d_token),
                'attn': nn.MultiheadAttention(d_token, n_heads, dropout=attention_dropout, batch_first=True),
                'norm2': nn.LayerNorm(d_token),
                'ffn': nn.Sequential(
                    nn.Linear(d_token, d_ffn * 2),
                    ReGLU(),
                    nn.Dropout(ffn_dropout),
                    nn.Linear(d_ffn, d_token),
                    nn.Dropout(residual_dropout),
                )
            }))
            
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.ReLU(),
            nn.Linear(d_token, n_targets)
        )
        
    def forward(self, x):
        # x: (batch, n_features)
        x = self.tokenizer(x) # (batch, n_features, d_token)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        
        for layer in self.layers:
            # Pre-Norm Attention
            x_norm = layer['norm1'](x)
            attn_out, _ = layer['attn'](x_norm, x_norm, x_norm)
            x = x + attn_out
            
            # Pre-Norm FFN
            x_norm = layer['norm2'](x)
            ffn_out = layer['ffn'](x_norm)
            x = x + ffn_out
            
        # Use CLS token for prediction
        return self.head(x[:, 0])

print("FT-Transformer Model Defined.")

In [ ]:
# 4.4.1 数据加载
print("Preparing data for optimization (Train + Test)...")

# Ensure we have the dataframes
if 'dae_df' not in locals():
    dae_df = pd.read_parquet('dae_features.parquet')
if 'knn_df' not in locals():
    knn_df = pd.read_parquet('knn_features.parquet')
# if 'sae_df' not in  locals():
#     sae_df = pd.read_parquet('sae_features.parquet')
# if 'cnn_df' not in  locals():
#     cnn_df = pd.read_parquet('cnn_features.parquet')
    
# Load Test Features
if 'dae_test_df' not in locals():
    dae_test_df = pd.read_parquet('dae_features_test.parquet')
if 'knn_test_df' not in locals():
    knn_test_df = pd.read_parquet('knn_features_test.parquet')
# if 'sae_test_df' not in locals():
#     sae_test_df = pd.read_parquet('sae_features_test.parquet')
# if 'cnn_test_df' not in  locals():
#     cnn_test_df = pd.read_parquet('cnn_features.parquet')
    
# Combine all features: Raw + KNN + DAE + SAE + CNN
# Construct final feature matrix
X_final = pd.concat([train_df[feature_cols], knn_df, dae_df], axis=1).values
y_final = train_df[target_cols].values

X_final_test = pd.concat([test_df[feature_cols], knn_test_df, dae_test_df], axis=1).values

print(f"Final Input Shape (Train): {X_final.shape}")
print(f"Final Input Shape (Test): {X_final_test.shape}")

# Move to GPU directly (H100 optimization)
X_tensor = torch.tensor(X_final, dtype=torch.float32).to(DEVICE)
y_tensor = torch.tensor(y_final, dtype=torch.float32).to(DEVICE)
X_test_tensor = torch.tensor(X_final_test, dtype=torch.float32).to(DEVICE)

# Check for BF16 support
try:
    if torch.cuda.is_bf16_supported():
        dtype_train = torch.bfloat16
        print("✅ BF16 (Bfloat16) supported and enabled.")
    else:
        dtype_train = torch.float16
        print("⚠️ BF16 not supported, falling back to FP16.")
except:
    dtype_train = torch.float32
    print("⚠️ BF16 check failed, using FP32.")

In [ ]:
import optuna
from optuna.trial import TrialState
import gc
import torch.nn as nn

print("4.4.3 Optuna 超参数搜索策略 (Competition Mode: Dual T4 GPU)")
print("=" * 80)

# Enable CuDNN benchmark for T4
torch.backends.cudnn.benchmark = True

def objective(trial):
    # Clear memory before each trial
    gc.collect()
    torch.cuda.empty_cache()

    # Hyperparameters Search Space
    # 降低 d_token 上限以防止 OOM，或者减小 Batch Size
    d_token = trial.suggest_categorical('d_token', [96, 192, 256, 384]) 
    n_layers = trial.suggest_int('n_layers', 1, 3) 
    n_heads = 8 
    attention_dropout = trial.suggest_float('attention_dropout', 0.1, 0.4)
    ffn_dropout = trial.suggest_float('ffn_dropout', 0.1, 0.4)
    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    d_ffn_factor = 1.33 
    
    # CV Loop (Competition: Use 3 folds for robust estimation)
    val_losses = []
    search_splits = splits[:1] 
    
    # 动态调整 Batch Size
    # 如果 d_token 较大，使用较小的 Batch Size
    if d_token >= 256:
        BATCH_SIZE = 1024
    else:
        BATCH_SIZE = 2048
        
    try:
        for fold_idx, (train_idx, val_idx) in enumerate(search_splits):
            # Model Initialization
            model = FTTransformer(
                n_num_features=X_final.shape[1],
                n_targets=3,
                d_token=d_token,
                n_layers=n_layers,
                n_heads=n_heads,
                d_ffn_factor=d_ffn_factor,
                attention_dropout=attention_dropout,
                ffn_dropout=ffn_dropout
            ).to(DEVICE)
            
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            
            optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
            criterion = nn.L1Loss()
            
            # Data Slicing
            X_tr, y_tr = X_tensor[train_idx], y_tensor[train_idx]
            X_val, y_val = X_tensor[val_idx], y_tensor[val_idx]
            
            # Training Loop
            model.train()
            n_batches = (len(X_tr) + BATCH_SIZE - 1) // BATCH_SIZE
            
            EPOCHS_SEARCH = 5
            
            for epoch in range(EPOCHS_SEARCH):
                perm = torch.randperm(len(X_tr), device=DEVICE)
                
                for i in range(n_batches):
                    idx = perm[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                    batch_x, batch_y = X_tr[idx], y_tr[idx]
                    
                    with torch.cuda.amp.autocast(dtype=dtype_train):
                        outputs = model(batch_x)
                        loss = criterion(outputs, batch_y)
                    
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    
                # Pruning check
                if fold_idx == 0:
                    model.eval()
                    val_loss_epoch = 0.0
                    val_steps = 0
                    n_val_batches = (len(X_val) + BATCH_SIZE - 1) // BATCH_SIZE
                    
                    with torch.no_grad():
                        for i in range(n_val_batches):
                            batch_x = X_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                            batch_y = y_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                            with torch.cuda.amp.autocast(dtype=dtype_train):
                                outputs = model(batch_x)
                                loss = criterion(outputs, batch_y)
                            val_loss_epoch += loss.item()
                            val_steps += 1
                    val_loss_epoch /= val_steps
                    
                    trial.report(val_loss_epoch, epoch)
                    if trial.should_prune():
                        raise optuna.exceptions.TrialPruned()
                    model.train()
            
            # Final Validation
            model.eval()
            val_loss_fold = 0.0
            val_steps = 0
            n_val_batches = (len(X_val) + BATCH_SIZE - 1) // BATCH_SIZE
            with torch.no_grad():
                for i in range(n_val_batches):
                    batch_x = X_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                    batch_y = y_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                    with torch.cuda.amp.autocast(dtype=dtype_train):
                        outputs = model(batch_x)
                        loss = criterion(outputs, batch_y)
                    val_loss_fold += loss.item()
                    val_steps += 1
            val_losses.append(val_loss_fold / val_steps)
            
            # Cleanup
            del model, optimizer, X_tr, y_tr, X_val, y_val
            gc.collect()
            torch.cuda.empty_cache()
            
    except torch.cuda.OutOfMemoryError:
        print(f"⚠️ Trial {trial.number} failed due to OOM. Pruning...")
        gc.collect()
        torch.cuda.empty_cache()
        raise optuna.exceptions.TrialPruned()
    except Exception as e:
        print(f"⚠️ Trial {trial.number} failed with error: {e}")
        raise e
        
    return sum(val_losses) / len(val_losses)

# Create Study
study = optuna.create_study(
    direction='minimize', 
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner()
)

print(f"Starting Optuna optimization on {torch.cuda.device_count()} GPUs...")
# Competition: Run more trials
study.optimize(objective, n_trials=10) 

print("Best params:", study.best_params)
print("Best value:", study.best_value)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import gc
import os

print("4.4.4 生成 FT-Transformer OOF 预测 (Final Optimized Run)")
print("=" * 80)

# T4 优化
torch.backends.cudnn.benchmark = True

# 1. 锁定最佳参数 (来自之前的 Trial 6)
# 直接写死，防止变量丢失或回退到默认值
if study.best_params:
    final_params = study.best_params
else:
    final_params = {
        'd_token': 256,
        'n_layers': 3,
        'attention_dropout': 0.2545,
        'ffn_dropout': 0.1383,
        'lr': 0.00034,  # 这是 Optuna 找到的最佳 LR
        'weight_decay': 2.53e-05
    }
print(f"✅ Using Locked Best Params: {final_params}")

# 2. 初始化容器
oof_ftt = np.zeros(y_final.shape)
test_preds_ftt = np.zeros((len(X_final_test), 3)) 

# 3. 训练配置
# T4 安全配置：Batch Size 1024
BATCH_SIZE = 1024 
EPOCHS = 20  # 设置为 20，配合 Early Stopping
EARLY_STOP_PATIENCE = 5

print(f"Starting 5-Fold CV Training (Batch: {BATCH_SIZE})...")

for fold, (train_idx, val_idx) in enumerate(splits):
    print(f"\n>>> Fold {fold + 1}/{len(splits)} <<<")
    
    # --- Data Loading (GPU Slice) ---
    X_tr, y_tr = X_tensor[train_idx], y_tensor[train_idx]
    X_val, y_val = X_tensor[val_idx], y_tensor[val_idx]
    
    # --- Model Init ---
    model = FTTransformer(
        n_num_features=X_final.shape[1],
        n_targets=3,
        d_token=final_params['d_token'],
        n_layers=final_params['n_layers'],
        n_heads=8,
        d_ffn_factor=1.33,
        attention_dropout=final_params['attention_dropout'],
        ffn_dropout=final_params['ffn_dropout']
    ).to(DEVICE)
    
    # Dual GPU Support
    if torch.cuda.device_count() > 1:
        print(f"   Using {torch.cuda.device_count()} GPUs (DataParallel)")
        model = nn.DataParallel(model)
    
    # --- Optimizer & Scheduler ---
    # 修正：max_lr 不要乘以 10，直接用搜索到的最佳值
    optimizer = optim.AdamW(model.parameters(), lr=final_params['lr'], weight_decay=final_params['weight_decay'])
    criterion = nn.L1Loss()
    
    steps_per_epoch = (len(X_tr) + BATCH_SIZE - 1) // BATCH_SIZE
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=final_params['lr'], # 保持与 Optuna 一致
        steps_per_epoch=steps_per_epoch,
        epochs=EPOCHS,
        pct_start=0.3  # 30% 时间热身
    )

    # --- Training Loop with Early Stopping ---
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    n_batches = steps_per_epoch
    
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(X_tr), device=DEVICE)
        train_loss_sum = 0
        
        for i in range(n_batches):
            idx = perm[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
            batch_x, batch_y = X_tr[idx], y_tr[idx]
            
            # BF16/FP16 混合精度
            with torch.cuda.amp.autocast(dtype=torch.float16):
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            train_loss_sum += loss.item()
            
        # Validation
        model.eval()
        val_loss_sum = 0
        n_val_batches = (len(X_val) + BATCH_SIZE - 1) // BATCH_SIZE
        
        with torch.no_grad():
            for i in range(n_val_batches):
                batch_x = X_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                batch_y = y_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                with torch.cuda.amp.autocast(dtype=torch.float16):
                    outputs = model(batch_x)
                    loss = criterion(outputs, batch_y)
                val_loss_sum += loss.item()
        
        avg_train_loss = train_loss_sum / n_batches
        avg_val_loss = val_loss_sum / n_val_batches
        
        print(f"   Epoch {epoch+1}/{EPOCHS} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}", end="")
        
        # Early Stopping Logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            # 保存最佳权重 (深拷贝到 CPU 内存以免占用显存)
            if isinstance(model, nn.DataParallel):
                best_model_state = {k: v.cpu().clone() for k, v in model.module.state_dict().items()}
            else:
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(" | ⭐ Best")
        else:
            patience_counter += 1
            print(f" | Patience {patience_counter}/{EARLY_STOP_PATIENCE}")
            
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"   ⏹️ Early stopping triggered at Epoch {epoch+1}")
            break
            
    # --- Load Best Weights & Inference ---
    print("   Restoring best model for inference...")
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_model_state)
    else:
        model.load_state_dict(best_model_state)
    
    model.eval()
    
    # 1. OOF Inference
    val_preds_fold = []
    with torch.no_grad():
        for i in range(n_val_batches):
            batch_x = X_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
            with torch.cuda.amp.autocast(dtype=torch.float16):
                preds = model(batch_x)
            val_preds_fold.append(preds.float().cpu().numpy())
    oof_ftt[val_idx] = np.vstack(val_preds_fold)
    
    # 2. Test Inference
    test_preds_fold = []
    n_test_batches = (len(X_test_tensor) + BATCH_SIZE - 1) // BATCH_SIZE
    with torch.no_grad():
        for i in range(n_test_batches):
            batch_x = X_test_tensor[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
            with torch.cuda.amp.autocast(dtype=torch.float16):
                preds = model(batch_x)
            test_preds_fold.append(preds.float().cpu().numpy())
            
    test_preds_ftt += np.vstack(test_preds_fold) / len(splits)
    
    # Cleanup
    del model, optimizer, scheduler, best_model_state, X_tr, y_tr, X_val, y_val
    gc.collect()
    torch.cuda.empty_cache()

print("\nSaving results...")
np.save('oof_ftt.npy', oof_ftt)
np.save('test_preds_ftt.npy', test_preds_ftt)
print("✅ FT-Transformer OOF & Test predictions saved.")

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

print("4.5 阶段五：GBDT 多目标回归 (XGBoost - High Noise Regime + Test Inference)")
print("=" * 80)

# 4.5.2 特征输入: 使用全量特征 (X_final)
print(f"GBDT Input Shape: {X_final.shape}")

# 初始化 OOF 容器
oof_xgb = np.zeros(y_final.shape)
test_preds_xgb = np.zeros((len(X_final_test), 3))

# 4.5.3 训练配置 (针对高噪声金融数据的极端防御配置)
xgb_params = {
    'tree_method': 'hist',
    'device': 'cuda',
    'objective': 'reg:squarederror', 
    'eval_metric': 'mae',
    
    # --- 核心抗噪参数 ---
    'learning_rate': 0.05, # 极低的学习率，步步为营
    'max_depth': 8,         # 极浅的树，只学大规律，不扣细节
    'min_child_weight': 200,# 一个叶子必须包含 200+ 样本，防止拟合个例
    'gamma': 0.1,           # 分裂所需的最小 Loss 减少量
    'base_score': 0.0,      # 收益率均值接近 0，显式指定
    
    # --- 正则化 ---
    'colsample_bytree': 0.5, # 每次只看一半特征
    'subsample': 0.6,        # 每次只看 60% 数据
    'reg_lambda': 20.0,      # 极强的 L2 正则
    'reg_alpha': 5.0,        # 增加 L1 正则，进行特征选择
    
    'n_estimators': 1000,    # 配合低学习率，增加树的数量
    'early_stopping_rounds': 500,
    'n_jobs': -1
}

targets = ['short', 'medium', 'long']

for i, target_name in enumerate(targets):
    print(f"\nTraining XGBoost for Target: {target_name}")
    
    for fold, (train_idx, val_idx) in enumerate(splits):
        # Prepare Data
        X_tr, y_tr = X_final[train_idx], y_final[train_idx, i]
        X_val, y_val = X_final[val_idx], y_final[val_idx, i]
        
        # Train
        model = xgb.XGBRegressor(**xgb_params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=1000 # 减少打印频率
        )
        
        # Predict OOF
        val_preds = model.predict(X_val)
        oof_xgb[val_idx, i] = val_preds
        
        # Predict Test (Accumulate)
        test_preds_xgb[:, i] += model.predict(X_final_test) / len(splits)
        
        # Check if model learned anything (Best iteration > 0)
        if model.best_iteration < 5:
            print(f"  ⚠️ Fold {fold+1}: Model failed to learn (Best Iter: {model.best_iteration}). Signal is too weak.")
        else:
            print(f"  ✅ Fold {fold+1}: Learned {model.best_iteration} trees. Best MAE: {model.best_score:.5f}")

print("\nXGBoost Training Complete.")
print("OOF MAE per target (Purged CV):")
mae_short = mean_absolute_error(y_final[:, 0], oof_xgb[:, 0])
mae_medium = mean_absolute_error(y_final[:, 1], oof_xgb[:, 1])
mae_long = mean_absolute_error(y_final[:, 2], oof_xgb[:, 2])
print(f"Short: {mae_short:.6f}")
print(f"Medium: {mae_medium:.6f}")
print(f"Long: {mae_long:.6f}")

In [ ]:
import numpy as np
import pandas as pd
import os

print("4.6 阶段六：手动加权集成 (Manual Weighted Ensemble) + 生成提交")
print("=" * 80)

# --- 1. 加载 FT-Transformer 的预测结果 ---
print("Loading FT-Transformer predictions...")
try:
    # 确保加载的是 Test 集的预测 (不是 OOF)
    test_preds_ftt = np.load('test_preds_ftt.npy')
    print(f"✅ Loaded FTT Test Preds: {test_preds_ftt.shape}")
except FileNotFoundError:
    print("❌ Error: 'test_preds_ftt.npy' not found! using zeros.")
    test_preds_ftt = np.zeros((len(test_df), 3))

# --- 2. 检查 XGBoost 的预测结果 ---
# 确保 test_preds_xgb 存在于当前内存中
if 'test_preds_xgb' not in locals():
    print("⚠️ Warning: 'test_preds_xgb' not found in memory. Checking file...")
    # 如果你有保存 xgb 的结果，可以在这里加载，否则初始化为0
    # test_preds_xgb = np.load('test_preds_xgb.npy') 
    test_preds_xgb = np.zeros_like(test_preds_ftt)
else:
    print(f"✅ Found XGB Test Preds: {test_preds_xgb.shape}")

# --- 3. 配置集成权重 ---
# 既然 XGBoost 单模 (0.7956) 比之前的 Stacking (0.8026) 好
# 说明 XGB 的贡献被低估了。我们采用你建议的 50/50 混合方案
# 你也可以根据 Public LB 的反馈微调这里的比例
W_FTT = 0
W_XGB = 1

print(f"\nEnsemble Strategy: {W_FTT} * FTT + {W_XGB} * XGB")
print("-" * 50)

# --- 4. 执行加权平均 ---
final_test_preds = np.zeros((len(test_preds_xgb), 3))
targets = ['target_short', 'target_medium', 'target_long']

# for i, target_name in enumerate(targets):
#     # 简单的线性加权
#     pred_ftt = test_preds_ftt[:, i]
#     pred_xgb = test_preds_xgb[:, i]
    
#     # 混合
#     weighted_pred = (W_FTT * pred_ftt) + (W_XGB * pred_xgb)
    
#     final_test_preds[:, i] = weighted_pred
    
#     # 打印一些统计信息，确保没有出现量级偏差
#     print(f"Target: {target_name:<10}")
#     print(f"  FTT Mean: {pred_ftt.mean():.6f} | XGB Mean: {pred_xgb.mean():.6f}")
#     print(f"  Ensembled Mean: {weighted_pred.mean():.6f}")
#     print("-" * 30)

for i, target_name in enumerate(targets):
    # 简单的线性加权
    pred_ftt = test_preds_ftt[:, i]
    pred_xgb = test_preds_xgb[:, i]

    if i==0:
        W_FTT,W_XGB = 1,0
    elif i==1:
        W_FTT,W_XGB = 0.5,0.5
    elif i==2:
        W_FTT,W_XGB = 0,1
    # 混合
    weighted_pred = (W_FTT * pred_ftt) + (W_XGB * pred_xgb)
    
    final_test_preds[:, i] = weighted_pred
    
    # 打印一些统计信息，确保没有出现量级偏差
    print(f"Target: {target_name:<10}")
    print(f"  FTT Mean: {pred_ftt.mean():.6f} | XGB Mean: {pred_xgb.mean():.6f}")
    print(f"  Ensembled Mean: {weighted_pred.mean():.6f}")
    print("-" * 30)

# --- 5. 生成提交文件 ---
print("\nGenerating Submission File...")

# 确保 ID 列存在
if 'id' in test_df.columns:
    ids = test_df['id']
else:
    ids = test_df.index

submission = pd.DataFrame({
    'id': ids,
    'target_short': final_test_preds[:, 0],
    'target_medium': final_test_preds[:, 1],
    'target_long': final_test_preds[:, 2]
})

submission_file = 'submission.csv'
submission.to_csv(submission_file, index=False)
print(f"✅ Submission saved to: {submission_file}")
print(submission.head())

# 2. XGBoost+MLP

### 1. feature engineering

In [ ]:
# NOTE: Run this cell before training XGBoost-based models.
from typing import List

print("Applying feature engineering for XGBoost (stationarity, time features, interactions)...")

# Helper configuration
feature_6_min = min(train_df['feature_6'].min(), test_df['feature_6'].min()) if 'feature_6' in train_df.columns else 0
feature_6_max = max(train_df['feature_6'].max(), test_df['feature_6'].max()) if 'feature_6' in train_df.columns else 0

# Columns to use when computing group-wise differences (if available)
group_candidates: List[str] = [
    col for col in ['date_id', 'time_id', 'stock_id'] if col in train_df.columns
]

# Ensure feature column list exists
if 'feature_cols' not in globals():
    feature_cols = [c for c in train_df.columns if c.startswith('feature_')]

# 1. Stationary transformation for feature_11 -> f11_diff (drop original)
for df_name, df in [('train', train_df), ('test', test_df)]:
    if 'feature_11' in df.columns:
        if group_candidates:
            df['f11_diff'] = df.groupby(group_candidates)['feature_11'].diff()
        else:
            df['f11_diff'] = df['feature_11'].diff()
        df['f11_diff'].fillna(0.0, inplace=True)
        df.drop(columns='feature_11', inplace=True)
    elif 'f11_diff' not in df.columns:
        df['f11_diff'] = 0.0

# 2. Drop low-value raw feature_27
for df in [train_df, test_df]:
    if 'feature_27' in df.columns:
        df.drop(columns='feature_27', inplace=True)

# 3. Intraday time derivatives from feature_6
if 'feature_6' in train_df.columns:
    for df_name, df in [('train', train_df), ('test', test_df)]:
        df['dist_to_open'] = df['feature_6'] - feature_6_min
        df['dist_to_close'] = feature_6_max - df['feature_6']
        df['is_midday'] = ((df['feature_6'] > 85) & (df['feature_6'] < 190)).astype(int)
        df['volatility_time_norm'] = df['feature_4'] / (df['feature_6'] - feature_6_min + 10)
        df['late_session_flow'] = df['feature_4'] * (df['feature_6'] > 200).astype(int)

# 4. Log transforms to spread long-tailed signals
for col in ['feature_4', 'feature_13', 'feature_14']:
    if col in train_df.columns:
        train_df[f'{col}_log1p'] = np.log1p(np.clip(train_df[col], a_min=0, a_max=None))
        test_df[f'{col}_log1p'] = np.log1p(np.clip(test_df[col], a_min=0, a_max=None))

# 5. Ratio-style features for risk/liquidity interpretation
if 'feature_4' in train_df.columns and 'f11_diff' in train_df.columns:
    denom = lambda x: x.replace(0, np.nan)
    train_df['f11_diff_div_f4'] = train_df['f11_diff'] / denom(train_df['feature_4'])
    test_df['f11_diff_div_f4'] = test_df['f11_diff'] / denom(test_df['feature_4'])
    train_df['f11_diff_div_f4'].replace([np.inf, -np.inf], 0.0, inplace=True)
    test_df['f11_diff_div_f4'].replace([np.inf, -np.inf], 0.0, inplace=True)
    train_df['f11_diff_div_f4'].fillna(0.0, inplace=True)
    test_df['f11_diff_div_f4'].fillna(0.0, inplace=True)

if 'feature_13' in train_df.columns and 'feature_4' in train_df.columns:
    denom = lambda x: x.replace(0, np.nan)
    train_df['feature_13_div_f4'] = train_df['feature_13'] / denom(train_df['feature_4'])
    test_df['feature_13_div_f4'] = test_df['feature_13'] / denom(test_df['feature_4'])
    for df in [train_df, test_df]:
        df['feature_13_div_f4'].replace([np.inf, -np.inf], 0.0, inplace=True)
        df['feature_13_div_f4'].fillna(0.0, inplace=True)

# 6. Categorical handling for low-cardinality features
for cat_col in ['feature_1', 'feature_8', 'feature_12']:
    if cat_col in train_df.columns:
        train_df[cat_col] = train_df[cat_col].astype('category')
        test_df[cat_col] = test_df[cat_col].astype('category')

# Count encoding for feature_8 (using train distribution)
if 'feature_8' in train_df.columns:
    feature_8_counts = train_df['feature_8'].value_counts()
    for df in [train_df, test_df]:
        mapped_counts = df['feature_8'].map(feature_8_counts).astype('float64')
        df.drop(columns=['feature_8_count'], errors='ignore', inplace=True)
        df['feature_8_count'] = mapped_counts.fillna(0.0)

# 7. Update feature column registry
new_features = [
    'f11_diff', 'dist_to_open', 'dist_to_close', 'is_midday',
    'volatility_time_norm', 'late_session_flow',
    'feature_4_log1p', 'feature_13_log1p', 'feature_14_log1p',
    'f11_diff_div_f4', 'feature_13_div_f4', 'feature_8_count'
]

dropped_features = ['feature_11', 'feature_27']
feature_cols = [col for col in feature_cols if col not in dropped_features]
for extra in new_features:
    if extra not in feature_cols:
        feature_cols.append(extra)

print(f"Updated feature set size: {len(feature_cols)}")

# XGBoost+MLP

In [16]:
split_idx = int(len(train_df) * 0.8)
print(f"Train/Val Split Index: {split_idx}")

X_train = train_df[feature_cols].iloc[:split_idx].values
y_train = train_df[target_cols].iloc[:split_idx].values
X_val = train_df[feature_cols].iloc[split_idx+480:].values
y_val = train_df[target_cols].iloc[split_idx+480:].values
X_test = test_df[feature_cols].values

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train/Val Split Index: 111513
Train: (111513, 29), Val: (27399, 29), Test: (34348, 29)


In [8]:
print("=" * 80)
print("使用 Optuna 调优 XGBoost 参数（优化 weighted MAE）")
print("=" * 80)

import optuna
import xgboost as xgb
import torch
from sklearn.metrics import mean_absolute_error

TARGET_ORDER = ["short", "medium", "long"]

def objective(trial):
    params = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "device": "cuda" if torch.cuda.is_available() else "cpu",

        # Optuna 搜索空间
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 400, 1600),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 5.0, log=True),

        "eval_metric": "mae",
        "random_state": 42,
    }

    maes = []

    for idx, name in enumerate(TARGET_ORDER):
        model = xgb.XGBRegressor(**params)

        model.fit(
            X_train,
            y_train[:, idx],
            eval_set=[(X_val, y_val[:, idx])],
            verbose=False
        )

        val_pred = model.predict(X_val)
        maes.append(mean_absolute_error(y_val[:, idx], val_pred))

    weighted_mae = (
        TARGET_WEIGHTS["short"] * maes[0]
        + TARGET_WEIGHTS["medium"] * maes[1]
        + TARGET_WEIGHTS["long"] * maes[2]
    )

    return weighted_mae


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

best_params = study.best_params
print(f"\n最佳 weighted MAE: {study.best_value:.6f}")
print("最佳参数:")
for k, v in best_params.items():
    print(f"  {k}: {v}")


print("\n" + "=" * 80)
print("使用最佳参数训练 XGBoost 模型")
print("=" * 80)

xgb_models = {}
xgb_val_predictions = {}

fixed_params = {
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "eval_metric": "mae",
    "random_state": 42,
}

final_params = {**fixed_params, **best_params}

for idx, target_name in enumerate(TARGET_ORDER):
    print(f"\n--- 训练 target_{target_name} 的 XGBoost ---")

    model = xgb.XGBRegressor(**final_params)
    model.fit(
        X_train,
        y_train[:, idx],
        eval_set=[(X_val, y_val[:, idx])],
        verbose=100
    )

    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val[:, idx], val_pred)

    print(f"验证集 MAE: {val_mae:.6f}")

    xgb_models[target_name] = model
    xgb_val_predictions[target_name] = val_pred


xgb_weighted_mae = (
    TARGET_WEIGHTS["short"] * mean_absolute_error(y_val[:, 0], xgb_val_predictions["short"])
    + TARGET_WEIGHTS["medium"] * mean_absolute_error(y_val[:, 1], xgb_val_predictions["medium"])
    + TARGET_WEIGHTS["long"] * mean_absolute_error(y_val[:, 2], xgb_val_predictions["long"])
)

print("\n" + "="*80)
print("XGBoost 调优后总结")
print("="*80)
print(
    f"Short MAE:  {mean_absolute_error(y_val[:, 0], xgb_val_predictions['short']):.6f} (权重: {TARGET_WEIGHTS['short']})"
)
print(
    f"Medium MAE: {mean_absolute_error(y_val[:, 1], xgb_val_predictions['medium']):.6f} (权重: {TARGET_WEIGHTS['medium']})"
)
print(
    f"Long MAE:   {mean_absolute_error(y_val[:, 2], xgb_val_predictions['long']):.6f} (权重: {TARGET_WEIGHTS['long']})"
)
print(f"Weighted MAE: {xgb_weighted_mae:.6f}")
print("="*80)


[I 2026-01-02 19:44:54,693] A new study created in memory with name: no-name-0475322e-64b6-43db-90c4-dbc44a2a20b8


使用 Optuna 调优 XGBoost 参数（优化 weighted MAE）


Best trial: 0. Best value: 0.0105347:   5%|▌         | 1/20 [00:04<01:24,  4.43s/it]

[I 2026-01-02 19:44:59,121] Trial 0 finished with value: 0.010534714895905566 and parameters: {'max_depth': 4, 'learning_rate': 0.017404693069360242, 'n_estimators': 1296, 'subsample': 0.8664196077511197, 'colsample_bytree': 0.8648260254819392, 'min_child_weight': 0.9743512005528657, 'reg_alpha': 0.008745206345866301, 'reg_lambda': 0.02618593616056991}. Best is trial 0 with value: 0.010534714895905566.


Best trial: 0. Best value: 0.0105347:  10%|█         | 2/20 [00:13<02:12,  7.38s/it]

...

Best trial: 11. Best value: 0.00690977:  95%|█████████▌| 19/20 [01:53<00:04,  4.39s/it]

[I 2026-01-02 19:46:47,775] Trial 18 finished with value: 0.00897071260717584 and parameters: {'max_depth': 5, 'learning_rate': 0.006210780299880616, 'n_estimators': 774, 'subsample': 0.7445353911561563, 'colsample_bytree': 0.8681801952919928, 'min_child_weight': 1.5055682015936935, 'reg_alpha': 4.520045745230556e-05, 'reg_lambda': 0.0013013514536187986}. Best is trial 11 with value: 0.006909769728094558.


Best trial: 11. Best value: 0.00690977: 100%|██████████| 20/20 [01:54<00:00,  5.73s/it]

[I 2026-01-02 19:46:49,339] Trial 19 finished with value: 0.006942409149604631 and parameters: {'max_depth': 3, 'learning_rate': 0.0011298884084341133, 'n_estimators': 513, 'subsample': 0.6547592719806133, 'colsample_bytree': 0.8224944379081864, 'min_child_weight': 4.940313276904486, 'reg_alpha': 0.00026744804305165067, 'reg_lambda': 6.806759714053179e-05}. Best is trial 11 with value: 0.006909769728094558.

最佳 weighted MAE: 0.006910
最佳参数:
  max_depth: 3
  learning_rate: 0.0010348569308049377
  n_estimators: 400
  subsample: 0.659839028672512
  colsample_bytree: 0.9995438749244691
  min_child_weight: 6.051989966450393
  reg_alpha: 0.0004969858027670799
  reg_lambda: 1.4794680953786788e-05

使用最佳参数训练 XGBoost 模型

--- 训练 target_short 的 XGBoost ---
[0]	validation_0-mae:0.00284
[100]	validation_0-mae:0.00284


[200]	validation_0-mae:0.00284
[300]	validation_0-mae:0.00285
[399]	validation_0-mae:0.00285
验证集 MAE: 0.002848

--- 训练 target_medium 的 XGBoost ---
[0]	validation_0-mae:0.00741
[100]	validation_0-mae:0.00743
[200]	validation_0-mae:0.00748
[300]	validation_0-mae:0.00754
[399]	validation_0-mae:0.00761
验证集 MAE: 0.007608

--- 训练 target_long 的 XGBoost ---
[0]	validation_0-mae:0.01590
[100]	validation_0-mae:0.01581
[200]	validation_0-mae:0.01584
[300]	validation_0-mae:0.01592
[399]	validation_0-mae:0.01602
验证集 MAE: 0.016017

XGBoost 调优后总结
Short MAE:  0.002848 (权重: 0.5)
Medium MAE: 0.007608 (权重: 0.3)
Long MAE:   0.016017 (权重: 0.2)
Weighted MAE: 0.006910


In [10]:
print(pd.Series(val_pred).describe())

count    27399.000000
mean         0.001947
std          0.004551
min         -0.003174
25%         -0.000638
50%         -0.000154
75%          0.002446
max          0.021904
dtype: float64


In [11]:
# Standardize features for neural network (don't scale targets)
print("Standardizing features...")
from sklearn.preprocessing import RobustScaler

scaler_X = RobustScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

print("Standardization complete")

class MultiTargetDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y) if y is not None else None
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# Create datasets (use original y_train, y_val - no scaling)
train_dataset = MultiTargetDataset(X_train_scaled, y_train)
val_dataset = MultiTargetDataset(X_val_scaled, y_val)
test_dataset = MultiTargetDataset(X_test_scaled)

# Create dataloaders
BATCH_SIZE = 512*8*8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

class MLPMultiTarget(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64], num_targets=3, dropout=0.2):
        super(MLPMultiTarget, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        self.shared_layers = nn.Sequential(*layers)
        
        # Output layer: predict all 3 targets simultaneously
        self.output_layer = nn.Linear(prev_dim, num_targets)
    
    def forward(self, x):
        x = self.shared_layers(x)
        return self.output_layer(x)

# Initialize model
INPUT_DIM = X_train_scaled.shape[1]
mlp_model = MLPMultiTarget(input_dim=INPUT_DIM).to(DEVICE)

print(f"MLP Model:")
print(mlp_model)
print(f"\nTotal parameters: {sum(p.numel() for p in mlp_model.parameters()):,}")

Standardizing features...
Standardization complete
Train batches: 4
Validation batches: 1
Test batches: 2
MLP Model:
MLPMultiTarget(
  (shared_layers): Sequential(
    (0): Linear(in_features=29, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (output_layer): Linear(in_features=64, out_features=3, bias=True)
)

Total parameters: 12,675


In [12]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 2:
                X_batch, y_batch = batch
                X_batch = X_batch.to(device)
                
                outputs = model(X_batch)
                
                all_preds.append(outputs.cpu().numpy())
                all_labels.append(y_batch.numpy())
            else:
                X_batch = batch.to(device)
                outputs = model(X_batch)
                all_preds.append(outputs.cpu().numpy())
    
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels) if len(all_labels) > 0 else None
    
    return all_preds, all_labels

print("Training functions defined")

Training functions defined


In [13]:
print("Training MLP model...")
print("="*80)

criterion = nn.L1Loss()  # Use MAE loss directly
optimizer = optim.AdamW(mlp_model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

EPOCHS = 20
best_weighted_mae = float('inf')
patience = 5
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss = train_epoch(mlp_model, train_loader, criterion, optimizer, DEVICE)
    val_preds, val_labels = eval_epoch(mlp_model, val_loader, DEVICE)
    
    # Calculate MAE for each target
    mae_short = mean_absolute_error(val_labels[:, 0], val_preds[:, 0])
    mae_medium = mean_absolute_error(val_labels[:, 1], val_preds[:, 1])
    mae_long = mean_absolute_error(val_labels[:, 2], val_preds[:, 2])
    
    # Calculate weighted MAE
    weighted_mae = (
        TARGET_WEIGHTS['short'] * mae_short +
        TARGET_WEIGHTS['medium'] * mae_medium +
        TARGET_WEIGHTS['long'] * mae_long
    )
    
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(weighted_mae)
    new_lr = optimizer.param_groups[0]['lr']
    
    if new_lr != old_lr:
        print(f"  Learning rate reduced: {old_lr:.6f} -> {new_lr:.6f}")
    
    if weighted_mae < best_weighted_mae:
        best_weighted_mae = weighted_mae
        patience_counter = 0
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}:")
        print(f"  Train Loss: {train_loss:.6f}")
        print(f"  Val Weighted MAE: {weighted_mae:.6f} (Best: {best_weighted_mae:.6f})")
        print(f"    Short: {mae_short:.6f}, Medium: {mae_medium:.6f}, Long: {mae_long:.6f}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nMLP training complete")
print(f"Best validation Weighted MAE: {best_weighted_mae:.6f}")

Training MLP model...
Epoch 10/20:
  Train Loss: 0.106293
  Val Weighted MAE: 0.034077 (Best: 0.034077)
    Short: 0.032555, Medium: 0.037742, Long: 0.032386
Epoch 20/20:
  Train Loss: 0.061910
  Val Weighted MAE: 0.013547 (Best: 0.013547)
    Short: 0.011504, Medium: 0.013798, Long: 0.018275

MLP training complete
Best validation Weighted MAE: 0.013547


In [14]:
val_preds, val_labels = eval_epoch(mlp_model, val_loader, DEVICE)

mlp_weighted_mae = (
    TARGET_WEIGHTS['short'] * mean_absolute_error(val_labels[:, 0], val_preds[:, 0]) +
    TARGET_WEIGHTS['medium'] * mean_absolute_error(val_labels[:, 1], val_preds[:, 1]) +
    TARGET_WEIGHTS['long'] * mean_absolute_error(val_labels[:, 2], val_preds[:, 2])
)

print("="*80)
print("MLP Summary")
print("="*80)
print(f"Short MAE:  {mean_absolute_error(val_labels[:, 0], val_preds[:, 0]):.6f} (weight: 0.5)")
print(f"Medium MAE: {mean_absolute_error(val_labels[:, 1], val_preds[:, 1]):.6f} (weight: 0.3)")
print(f"Long MAE:   {mean_absolute_error(val_labels[:, 2], val_preds[:, 2]):.6f} (weight: 0.2)")
print(f"Weighted MAE: {mlp_weighted_mae:.6f}")
print("="*80)

MLP Summary
Short MAE:  0.011504 (weight: 0.5)
Medium MAE: 0.013798 (weight: 0.3)
Long MAE:   0.018275 (weight: 0.2)
Weighted MAE: 0.013547


In [15]:
# Compare models
comparison = pd.DataFrame({
    'Model': ['XGBoost (3 models)', 'MLP (multi-target)'],
    'Weighted MAE': [xgb_weighted_mae, mlp_weighted_mae]
})

print(comparison.to_string(index=False))

best_model = 'XGBoost' if xgb_weighted_mae < mlp_weighted_mae else 'MLP'
print(f"\nBest model: {best_model}")

             Model  Weighted MAE
XGBoost (3 models)      0.006910
MLP (multi-target)      0.013547

Best model: XGBoost


In [9]:
print("Generating XGBoost predictions...")
test_pred_xgb_short = xgb_models['short'].predict(X_test)
test_pred_xgb_medium = xgb_models['medium'].predict(X_test)
test_pred_xgb_long = xgb_models['long'].predict(X_test)

submission_xgb = pd.DataFrame({
    'id': test_df['id'],
    'target_short': test_pred_xgb_short,
    'target_medium': test_pred_xgb_medium,
    'target_long': test_pred_xgb_long
})
submission_xgb.to_csv('submission_xgb.csv', index=False)
print(f"✅ XGBoost submission saved: submission_xgb.csv")

# print("\nGenerating MLP predictions...")
# test_preds_mlp, _ = eval_epoch(mlp_model, test_loader, DEVICE)
# submission_mlp = pd.DataFrame({
#     'id': test_df['id'],
#     'target_short': test_preds_mlp[:, 0],
#     'target_medium': test_preds_mlp[:, 1],
#     'target_long': test_preds_mlp[:, 2]
# })
# submission_mlp.to_csv('submission.csv', index=False)
# print(f"✅ MLP submission saved: submission_mlp.csv")

Generating XGBoost predictions...
✅ XGBoost submission saved: submission_xgb.csv
